# Prompt Evaluator — Distributional Evaluation of System Prompts

**PromptEvaluator** scores a system prompt by running it **K times per query** and measuring how stable the response distribution is.

## Metrics

| Metric | Type | Description |
|--------|------|-------------|
| **CSR** (Consistency/Stability Rate) | Always active | Fraction of K responses in the dominant semantic cluster |
| **Stability** (1 - Semantic Entropy) | Always active | How focused the distribution of meanings is |
| **RSS** (Reference Similarity Score) | Optional — auto | Avg cosine similarity to reference answer |
| **ICR** (Instruction Compliance Rate) | Optional — auto | Fraction of verifiable constraints satisfied |
| **JQ** (Judge Quality) | Optional — manual | LLM-as-judge score (`jq_enabled=True`) |

> **Why K > 1?** A single model call hides variance. Running K times at `temperature > 0` reveals whether the prompt produces a focused distribution or scattered, unpredictable responses.

> **Key insight:** CSR=1.0 on closed factual questions is expected and correct — the prompt is doing its job. The interesting signal is on **open-ended or ambiguous questions** where a vague prompt will show CSR < 1.0.

## Installation

Install Fair Forge with prompt-evaluator support, a LangChain provider, and a sentence embedding model:

In [ ]:
!pip install "alquimia-fair-forge[prompt-evaluator]" langchain-groq sentence-transformers -q

## Imports

In [1]:
import getpass
import json
from pathlib import Path

from langchain_groq import ChatGroq

from fair_forge import Retriever
from fair_forge.embedders.sentence_transformer import SentenceTransformerEmbedder
from fair_forge.metrics.constraints import JsonConstraint, KeywordConstraint, WordCountConstraint
from fair_forge.metrics.prompt_evaluator import PromptEvaluator
from fair_forge.schemas import Batch, Dataset

## API Key

This example uses [Groq](https://console.groq.com) — a fast, free LLM API. Create a free account to get your key.

In [2]:
GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")

## Configuration

Define the prompt you want to evaluate and the sampling parameters.

- `K` — number of responses generated per query. Higher K = more reliable estimates. Default: 10
- `TAU` — cosine similarity threshold for clustering. Range 0.75–0.85 is recommended. Default: 0.80

In [ ]:
# The prompt under evaluation — intentionally underspecified to show variance
SEED_PROMPT = (
    "You are an assistant for Nexo. Answer questions about our product."
)

K   = 5     # samples per query 
TAU = 0.93  # clustering threshold

## Dataset

Using open-ended questions where a vague prompt produces genuine variation across K runs.
These questions require synthesis or recommendation — the model can take different angles each time.

Add `ground_truth_assistant` to activate RSS automatically.

## Retriever

Implement a `Retriever` subclass to load your dataset. The fallback data below is used when no JSON file is found.

In [11]:
class MyRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        context = (
            "Nexo offers three plans: Free (up to 3 projects, 5 members), "
            "Pro ($15/month per user, unlimited projects, priority support), "
            "and Business ($25/month per user, SSO, dedicated account manager). "
            "To reset your password, click 'Forgot your password?' on the login screen "
            "and enter your email. A reset link will be sent within 2 minutes."
        )
        return [
            Dataset(
                session_id="nexo-eval",
                assistant_id="support-bot-v1",
                language="english",
                context=context,
                conversation=[
                    Batch(
                        qa_id="q1",
                        query="What plan would you recommend for a small startup?",
                        assistant="",
                        ground_truth_assistant="For a small startup, the Free plan covers up to 3 projects and 5 members at no cost. If you need more, the Pro plan is $15 per user per month.",
                    ),
                    Batch(
                        qa_id="q2",
                        query="What are the main advantages of upgrading from Free to Pro?",
                        assistant="",
                        ground_truth_assistant="Upgrading to Pro gives you unlimited projects, priority support, and costs $15 per user per month.",
                    ),
                    Batch(
                        qa_id="q3",
                        query="How do I choose between Pro and Business?",
                        assistant="",
                        ground_truth_assistant="Choose Pro at $15/month for unlimited projects and priority support. Choose Business at $25/month if you need SSO and a dedicated account manager.",
                    ),
                ],
            )
        ]

## Model & Embedder

- **model** — the LLM that executes the prompt K times per query. Needs `temperature > 0`.
- **embedder** — encodes responses into vectors for semantic clustering and RSS computation.

In [16]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.9,  # must be > 0 for K samples to vary
)

embedder = SentenceTransformerEmbedder(model_name="all-MiniLM-L6-v2")

2026-03-25 13:38:31,664 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-03-25 13:38:31,665 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


## Run Evaluation

`PromptEvaluator.run()` handles everything internally — K model calls, embedding, clustering, and metric computation.

RSS is activated automatically because `ground_truth_assistant` is present in the dataset.

In [17]:
results = PromptEvaluator.run(
    MyRetriever,
    model=model,
    seed_prompt=SEED_PROMPT,
    embedder=embedder,
    k=K,
    tau=TAU,
    verbose=True,
)

2026-03-25 13:38:35,077 - fair_forge.utils.logging - INFO - Starting to process dataset
2026-03-25 13:38:35,078 - fair_forge.utils.logging - INFO - Processing using dataset/session level parsing
2026-03-25 13:38:35,079 - fair_forge.utils.logging - INFO - Session ID: nexo-eval, Assistant ID: support-bot-v1
2026-03-25 13:38:35,079 - fair_forge.utils.logging - DEBUG - QA ID: q1
2026-03-25 13:38:35,886 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:38:36,810 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:38:37,871 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:38:39,049 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:38:40,113 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 

## Results

`print(result)` shows all active metrics and explains each one.

With an underspecified prompt and open-ended questions you should see **CSR and Stability below 1.0** — the model is taking different angles on the same question.

Compare this with a tighter prompt (e.g. adding format and scope instructions) and CSR should increase.

In [18]:
print(results[0])

 PROMPT EVALUATOR

 METRICS
 ──────────────────────────────────────────────────────────
  CSR  (Consistency/Stability Rate)          always active
       How often the model gives the same meaning across K runs.
       1.0 = perfectly consistent  |  0.0 = completely scattered

  Stability  (1 - Semantic Entropy)          always active
       How focused the distribution of meanings is.
       1.0 = all responses converge  |  0.0 = maximum spread

  RSS  (Reference Similarity Score)          optional — auto
       Similarity between responses and the reference answer.
       Activated automatically when ground_truth_assistant is in your dataset.

  ICR  (Instruction Compliance Rate)         optional — auto
       Fraction of verifiable prompt constraints satisfied per response.
       Activated automatically when constraints are passed to PromptEvaluator.

  JQ   (Judge Quality)                       optional — manual
       LLM-as-judge score per response.
       Activated with: jq_ena

## Activating ICR (Instruction Compliance Rate)

### When to use it

Use ICR when your prompt tells the model **how** to respond — not just what to say, but in what format, length, or structure.

If your prompt says things like:
- *"Always respond in JSON"*
- *"Keep your answer under 100 words"*
- *"Always include the word 'source'"*

...those are programmatically verifiable constraints. ICR measures whether the model consistently follows them across K runs.

If your prompt has no explicit formatting rules, skip ICR — CSR and RSS already cover what you need.

### Built-in constraints

| Constraint | What it checks |
|------------|----------------|
| `JsonConstraint()` | Response is valid JSON |
| `WordCountConstraint(max_words=N)` | Response has at most N words |
| `KeywordConstraint(keyword, case_sensitive=False)` | Response contains the keyword |
| `RegexConstraint(pattern)` | Response matches a regex pattern |

You can also write your own — any class with a `check(response: str) -> bool` method works.

In [19]:
results_icr = PromptEvaluator.run(
    MyRetriever,
    model=model,
    seed_prompt=(
        "You are an assistant for Nexo. "
        "Always respond in valid JSON with keys 'answer' and 'confidence'. "
        "Keep your answer under 40 words."
    ),
    embedder=embedder,
    k=K,
    tau=TAU,
    constraints=[
        JsonConstraint(),
        WordCountConstraint(max_words=40),
        KeywordConstraint("confidence"),
    ],
)

print(results_icr[0])

2026-03-25 13:39:12,666 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:39:13,534 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:39:14,037 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:39:14,473 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:39:14,983 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:39:15,543 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:39:16,361 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 13:39:16,874 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 

 PROMPT EVALUATOR

 METRICS
 ──────────────────────────────────────────────────────────
  CSR  (Consistency/Stability Rate)          always active
       How often the model gives the same meaning across K runs.
       1.0 = perfectly consistent  |  0.0 = completely scattered

  Stability  (1 - Semantic Entropy)          always active
       How focused the distribution of meanings is.
       1.0 = all responses converge  |  0.0 = maximum spread

  RSS  (Reference Similarity Score)          optional — auto
       Similarity between responses and the reference answer.
       Activated automatically when ground_truth_assistant is in your dataset.

  ICR  (Instruction Compliance Rate)         optional — auto
       Fraction of verifiable prompt constraints satisfied per response.
       Activated automatically when constraints are passed to PromptEvaluator.

  JQ   (Judge Quality)                       optional — manual
       LLM-as-judge score per response.
       Activated with: jq_ena

## Interpreting Results

### CSR — Consistency/Stability Rate

| Value | Interpretation |
|-------|----------------|
| 0.9–1.0 | Excellent — the prompt produces the same meaning almost every run |
| 0.7–0.9 | Good — occasional variation but a clear dominant response |
| 0.5–0.7 | Moderate — noticeable semantic scatter, review prompt clarity |
| < 0.5 | Poor — responses scattered across many different meanings |

### Stability — 1 - Semantic Entropy

| Value | Interpretation |
|-------|----------------|
| 0.9–1.0 | Excellent — all responses cluster tightly |
| 0.7–0.9 | Good — distribution has a clear mode |
| 0.5–0.7 | Moderate — multiple competing meaning groups |
| < 0.5 | Poor — high entropy, unpredictable output |

### CSR=1.0 is not always wrong

On closed factual questions ("How much does the Pro plan cost?") CSR=1.0 is **expected and correct** — there is only one right answer. Use RSS to validate correctness in that case.

CSR and Stability show their diagnostic value on **open-ended or synthesis questions**, where a vague prompt will produce scattered responses and a tight prompt will converge.

| CSR | RSS | What it means |
|-----|-----|---------------|
| High | High | Consistent and correct |
| High | Low | Consistently wrong — stable but hallucinating |
| Low | High | Correct but unpredictable — tighten the prompt |
| Low | Low | Scattered and wrong |

## Key Takeaways

1. **Use `temperature > 0`** — CSR and Stability need variance across K samples to be meaningful
2. **Use open-ended questions** — closed factual questions will always give CSR=1.0
3. **RSS is free** — add `ground_truth_assistant` to your dataset and it activates automatically
4. **ICR is free** — pass `constraints=[...]` with verifiable rules and it activates automatically
5. **Start with K=5** for iteration, increase to K=10-20 for final evaluation
6. **`print(result)` is all you need** — the output explains each metric and tells you how to activate the optional ones

### Resources
- [Fair Forge Documentation](https://docs.alquimia.ai)
- [Prompt Evaluator Docs](https://docs.alquimia.ai/metrics/prompt-evaluator)
- [GitHub](https://github.com/Alquimia-ai/fair-forge)